In [53]:
from flask import Flask, jsonify, request
import requests
from datetime import datetime
import re
import os
from src.prediction_pipeline.predict_output import (
    PredictionPipeline,
    CustomData
)
from dotenv import load_dotenv
load_dotenv()

def get_coordinates(location):
    API_KEY = os.getenv("OPENCAGE_API_KEY")

    url = "https://api.opencagedata.com/geocode/v1/json"

    params = {
        "q": location,
        "key": API_KEY,
        "limit": 1
    }

    response = requests.get(url, params=params)
    data = response.json()

    if response.status_code != 200:
        raise Exception(f"OpenCage API Error: {data}")

    if "results" not in data or not data["results"]:
        raise Exception("Location not found")

    lat = data["results"][0]["geometry"]["lat"]
    lng = data["results"][0]["geometry"]["lng"]

    return lat, lng


def classify_weather(x):
    s = str(x).lower().strip()

    if re.search(r"thunder|t-storm|storm", s):
        return "thunderstorm"

    elif re.search(r"snow|sleet|ice|wintry|freezing|blizzard", s):
        return "snow"

    elif re.search(r"rain|drizzle|shower|precipitation", s):
        return "rain"

    elif re.search(r"fog|mist|haze|smoke", s):
        return "fog"

    elif re.search(r"windy|squalls|blowing|sandstorm|dust", s):
        return "windy"

    elif re.search(r"clear|fair|sunny", s):
        return "clear"

    elif re.search(r"cloud|cloudy|overcast|partly cloudy|mostly cloudy", s):
        return "cloudy"

    return "cloudy"


def fetch_weather_by_coords(latitude, longitude, date, time):
    API_KEY = os.getenv("VISUAL_CROSSING_API_KEY")

    location = f"{latitude},{longitude}"

    url = (
        "https://weather.visualcrossing.com/"
        "VisualCrossingWebServices/rest/services/"
        f"timeline/{location}/{date}"
    )

    params = {
        "key": API_KEY,
        "include": "hours",
        "unitGroup": "metric"
    }

    response = requests.get(url, params=params)
    data = response.json()

    if response.status_code != 200:
        raise Exception(f"Visual Crossing API Error: {data}")

    if "days" not in data or not data["days"]:
        raise Exception("Weather data not available")

    target_hour = time[:2]

    weather = next(
        (
            h for h in data["days"][0]["hours"]
            if h["datetime"].startswith(target_hour)
        ),
        None
    )

    if weather is None:
        raise Exception("Weather unavailable for selected time")

    visibility_km = weather.get("visibility", 0)
    visibility_mi = round(visibility_km * 0.621371, 2)

    return {
        "Humidity(%)": weather["humidity"],
        "Visibility(mi)": visibility_mi,
        "Temperature(C)": weather["temp"],
        "Wind Speed(km/h)": weather["windspeed"],
        "Weather_Conditions":(weather["conditions"])
    }

In [54]:
lat,long=get_coordinates(location='Bengaluru, Karnataka')
fetch_weather_by_coords(latitude=lat,longitude=long,date='2026-05-25',time='15:00')

{'Humidity(%)': 35.08,
 'Visibility(mi)': 14.98,
 'Temperature(C)': 33.4,
 'Wind Speed(km/h)': 12.2,
 'Weather_Conditions': 'Clear'}

In [1]:
import pandas as pd


In [2]:
df=pd.read_csv('data_cleaned.csv')

In [4]:
df.head()

,Severity,Start_Lat,Start_Lng,StartTime,EndTime,Distance(mi),DelayFromTypicalTraffic(mins),Congestion_Speed,Street,City,...,State,Humidity(%),Visibility(mi),Weather_Event,Weather_Conditions,Temperature(C),Start_hour,Start_minute,Start_day,Start_month
0,2,39.191032,-120.819740,2016-12-21 00:19:00+00:00,2016-12-20T19:33:47.000-05:00,1.40,2.58,Moderate,I-80 W,Dutch Flat,...,CA,30.0,10.0,NaN,clear,12.222222,0.0,19.0,21.0,12.0
1,0,41.736015,-87.721565,2018-11-16 22:18:00+00:00,2018-11-16T18:08:28.000-05:00,0.73,0.42,Slow,S Pulaski Rd,Chicago,...,IL,70.0,10.0,NaN,cloudy,3.888889,22.0,18.0,16.0,11.0
2,0,32.519043,-93.741096,2021-02-19 01:32:00+00:00,2021-02-18T21:21:32.000-05:00,1.80,1.00,Moderate,E Texas St,Bossier City,...,LA,79.0,10.0,NaN,clear,-1.111111,1.0,32.0,19.0,2.0
3,0,40.730564,-74.001709,2020-11-13 13:06:00+00:00,2020-11-13T08:48:22.000-05:00,1.42,1.00,Slow,Avenue of the Americas,New York,...,NY,93.0,1.0,NaN,rain,8.888889,13.0,6.0,13.0,11.0
4,1,33.758331,-118.238533,2017-08-24 13:54:00+00:00,2017-08-24T11:13:19.000-04:00,2.60,4.90,Slow,W Ocean Blvd,Long Beach,...,CA,79.0,9.0,NaN,cloudy,19.388889,13.0,54.0,24.0,8.0


In [5]:
df.columns

Index(['Severity', 'Start_Lat', 'Start_Lng', 'StartTime', 'EndTime',
       'Distance(mi)', 'DelayFromTypicalTraffic(mins)', 'Congestion_Speed',
       'Street', 'City', 'County', 'State', 'Humidity(%)', 'Visibility(mi)',
       'Weather_Event', 'Weather_Conditions', 'Temperature(C)', 'Start_hour',
       'Start_minute', 'Start_day', 'Start_month'],
      dtype='object')

In [6]:
df1=df[['Street', 'City', 'County', 'State']]

In [9]:
df1

,Street,City,County,State
0,I-80 W,Dutch Flat,Placer,CA
1,S Pulaski Rd,Chicago,Cook,IL
2,E Texas St,Bossier City,Bossier,LA
3,Avenue of the Americas,New York,New York,NY
4,W Ocean Blvd,Long Beach,Los Angeles,CA
...,...,...,...,...
1999995,Pacheco Pass Hwy,Gilroy,Santa Clara,CA
1999996,CA-110 S,Los Angeles,Los Angeles,CA
1999997,South St,Bridgewater,Plymouth,MA
1999998,Concord Blvd,Concord,Contra Costa,CA


In [19]:
df1[(df1["Street"]=='I-80 W') & (df1['City']=='Dutch Flat')]

,Street,City,County,State
0,I-80 W,Dutch Flat,Placer,CA


In [23]:
df1.drop_duplicates(inplace=True)

C:\Users\CHINMAYA\AppData\Local\Temp\ipykernel_21764\4156330626.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1.drop_duplicates(inplace=True)


In [22]:
df1.drop(columns=['County'],inplace=True) 

C:\Users\CHINMAYA\AppData\Local\Temp\ipykernel_21764\179211625.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1.drop(columns=['County'],inplace=True)


In [25]:
df1.to_csv('places.csv',index=False)

In [ ]:
from flask import Flask, jsonify, request
import requests
from datetime import datetime
import re
import os
from src.prediction_pipeline.predict_output import (
    PredictionPipeline,
    CustomData
)
from dotenv import load_dotenv
load_dotenv()
app = Flask(__name__)


def get_coordinates(location):
    API_KEY = os.getenv("OPENCAGE_API_KEY")

    url = "https://api.opencagedata.com/geocode/v1/json"

    params = {
        "q": location,
        "key": API_KEY,
        "limit": 1
    }

    response = requests.get(url, params=params)
    data = response.json()

    if response.status_code != 200:
        raise Exception(f"OpenCage API Error: {data}")

    if "results" not in data or not data["results"]:
        raise Exception("Location not found")

    lat = data["results"][0]["geometry"]["lat"]
    lng = data["results"][0]["geometry"]["lng"]

    return lat, lng


def classify_weather(x):
    s = str(x).lower().strip()

    if re.search(r"thunder|t-storm|storm", s):
        return "thunderstorm"

    elif re.search(r"snow|sleet|ice|wintry|freezing|blizzard", s):
        return "snow"

    elif re.search(r"rain|drizzle|shower|precipitation", s):
        return "rain"

    elif re.search(r"fog|mist|haze|smoke", s):
        return "fog"

    elif re.search(r"windy|squalls|blowing|sandstorm|dust", s):
        return "windy"

    elif re.search(r"clear|fair|sunny", s):
        return "clear"

    elif re.search(r"cloud|cloudy|overcast|partly cloudy|mostly cloudy", s):
        return "cloudy"

    return "cloudy"


def fetch_weather_by_coords(latitude, longitude, date, time):
    API_KEY = os.getenv("VISUAL_CROSSING_API_KEY")

    location = f"{latitude},{longitude}"

    url = (
        "https://weather.visualcrossing.com/"
        "VisualCrossingWebServices/rest/services/"
        f"timeline/{location}/{date}"
    )

    params = {
        "key": API_KEY,
        "include": "hours",
        "unitGroup": "metric"
    }

    response = requests.get(url, params=params)
    data = response.json()

    if response.status_code != 200:
        raise Exception(f"Visual Crossing API Error: {data}")

    if "days" not in data or not data["days"]:
        raise Exception("Weather data not available")

    target_hour = time[:2]

    weather = next(
        (
            h for h in data["days"][0]["hours"]
            if h["datetime"].startswith(target_hour)
        ),
        None
    )

    if weather is None:
        raise Exception("Weather unavailable for selected time")

    visibility_km = weather.get("visibility", 0)
    visibility_mi = round(visibility_km * 0.621371, 2)

    return {
        "Humidity(%)": weather["humidity"],
        "Visibility(mi)": visibility_mi,
        "Temperature(C)": weather["temp"],
        "Wind Speed(km/h)": weather["windspeed"],
        "Weather_Conditions": classify_weather(weather["conditions"])
    }


@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "message": "Traffic Congestion Speed Prediction API Running"
    })


@app.route("/predictdata", methods=["POST"])
def predict():
    try:
        data = request.get_json()

        if not data:
            return jsonify({"error": "No JSON data provided"}), 400

        required_fields = [
            "location",
            "date",
            "time"
        ]

        missing_fields = [
            field for field in required_fields
            if field not in data
        ]

        if missing_fields:
            return jsonify({
                "error": f"Missing fields: {missing_fields}"
            }), 400

        location = data["location"]
        date = data["date"]
        time = data["time"]

        selected_date = datetime.strptime(date, "%Y-%m-%d").date()
        today = datetime.today().date()

        if selected_date < today:
            return jsonify({
                "error": "Past dates are not allowed. Enter today or future date."
            }), 400

        dt = datetime.strptime(
            f"{date} {time}",
            "%Y-%m-%d %H:%M"
        )

        lat, lng = get_coordinates(location)

        weather = fetch_weather_by_coords(
            latitude=lat,
            longitude=lng,
            date=date,
            time=time
        )

        input_data = CustomData(
            Start_Lat=lat,
            Start_Lng=lng,
            Humidity=weather["Humidity(%)"],
            Visibility=weather["Visibility(mi)"],
            Temperature=weather["Temperature(C)"],
            Start_hour=dt.hour,
            Start_minute=dt.minute,
            Start_month=dt.month,
            Start_day=dt.day,
            Weather_Conditions=weather["Weather_Conditions"],
            Severity=2
        )

        pred_df = input_data.get_datas_as_df()

        pipeline = PredictionPipeline()
        prediction = pipeline.predict(pred_df)

        prediction_map = {
            0: "Slow",
            1: "Moderate",
            2: "Fast"
        }

        prediction_result = prediction_map.get(
            int(prediction[0]),
            "Unknown"
        )

        return jsonify({
            "Traffic prediction": prediction_result,
            "coordinates": {
                "latitude": lat,
                "longitude": lng
            },
            "weather_used": weather,
            "status": "success"
        }), 200

    except ValueError as e:
        return jsonify({
            "error": f"Invalid datatype or date/time format: {str(e)}"
        }), 422

    except Exception as e:
        return jsonify({
            "error": f"Prediction failed: {str(e)}"
        }), 500


if __name__ == "__main__":
    app.run(
        debug=True,
        host="0.0.0.0",
        port=8000
    )